# 03 — Phase 1 Ready: Merged Dataset Validation

**Purpose:** Confirm the Phase 0.3 pipeline is fully wired before moving to Phase 1 feature engineering.

This notebook validates:
1. The merged 2025-26 dataset builds cleanly from BBRef CSV + salary CSV sources
2. Key advanced metrics (BPM, VORP, PER, WS) and salary fields are well-populated
3. The Lakers roster is fully enriched with contract information
4. A naive surplus-value proxy produces directionally sensible rankings
5. Luka Doncic's full merged row is correct

**Do not** re-run nba_api calls unnecessarily — the cache is already warm from Phase 0.1.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/suveerdhawan/Desktop/Codex/lakers-trade-engine


In [2]:
import logging

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(name)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)

from src.data.merge import build_player_dataset, completeness_summary
from src.data.config import CURRENT_SEASON

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_rows", 30)

print(f"Current season: {CURRENT_SEASON}")

Current season: 2025-26


## 1. Build the Merged Dataset

Pulls from:
- **nba_api** (cached parquet): base stats, advanced, shooting, defense
- **BBRef advanced CSV** (`data/raw/bbref_advanced_2025-26.csv`): PER, WS, BPM, VORP
- **BBRef salary CSV** (`data/raw/salaries_2025-26.csv`): multi-year contract detail

EPM is skipped; BPM serves as the primary efficiency anchor for Phase 1.

In [3]:
df = build_player_dataset(CURRENT_SEASON)
print(f"Dataset shape: {df.shape}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")

16:11:39  src.data.merge  INFO  Building player dataset for 2025-26 ...
16:11:39  src.data.cache  INFO  cache hit: nba_player_stats_2025-26_PerGame
16:11:39  src.data.cache  INFO  cache hit: nba_player_advanced_2025-26
16:11:39  src.data.cache  INFO  cache miss -- fetching: nba_shooting_catchshoot_2025-26
16:11:40  src.data.cache  INFO  cached 582 rows to nba_shooting_catchshoot_2025-26.parquet
16:11:40  src.data.cache  INFO  cache miss -- fetching: nba_shooting_pullup_2025-26
16:11:40  src.data.cache  INFO  cached 582 rows to nba_shooting_pullup_2025-26.parquet
16:11:40  src.data.cache  INFO  cache miss -- fetching: nba_player_defense_2025-26
16:11:41  src.data.cache  INFO  cached 581 rows to nba_player_defense_2025-26.parquet
16:11:41  src.data.bbref_stats  INFO  Loaded 734 BBRef advanced rows from CSV (2025-26)
16:11:41  src.data.salaries  INFO  Loaded 530 BBRef salary rows from salaries_2025-26.csv
16:11:41  src.data.merge  INFO  Dataset built: 582 players, 185 columns for 2025-26


Dataset shape: (582, 185)
Columns (185): ['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2', 'TD3', 'WNBA_FANTASY_PTS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK', 'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK', 'NBA_FANTASY_PTS_RANK', 'DD2_RANK', 'TD3_RANK', 'WNBA_FANTASY_PTS_RANK', 'TEAM_COUNT', 'E_OFF_RATING', 'OFF_RATING', 'sp_work_OFF_RATING', 'E_DEF_RATING', 'DEF_RATING', 'sp_work_DEF_RATING', 'E_NET_RATING', 'NET_RATING', 'sp_work_NET_RATING', 'AST_PCT', 'AST_TO', 'AST_RATIO', 'OREB_PCT', 'DREB_P

## 2. Completeness Report

BPM, VORP, PER, WS, SALARY, YEARS_REMAINING, and TOTAL_GUARANTEED should all be populated now.
Any source still at 0% indicates a join failure that needs investigation before Phase 1.

In [4]:
completeness_summary(df)

,source,sentinel_col,coverage_pct,n_players,n_total
0,nba_api_base,PTS,100.00,582,582
1,nba_api_advanced,NET_RATING,100.00,582,582
2,nba_api_defense,PCT_PLUSMINUS,99.80,581,582
3,bbref_advanced,BPM,99.30,578,582
4,bbref_ws,WS,99.30,578,582
5,bbref_per,PER,99.30,578,582
6,bbref_vorp,VORP,99.30,578,582
7,nba_api_shooting,CS_CATCH_SHOOT_FG3_PCT,93.60,545,582
8,salary_years,YEARS_REMAINING,80.60,469,582
9,salary_current,SALARY,80.40,468,582


## 3. Lakers Roster — Contract Snapshot

Who is expiring (trade asset / cap relief), who is locked up (trade cost), and what is the total guaranteed obligation.
This is the basis for the CBA salary-matching analysis in Phase 3.

In [5]:
lakers = df[df["TEAM_ABBREVIATION"] == "LAL"].copy()

salary_cols = ["PLAYER_NAME", "SALARY", "YEARS_REMAINING", "IS_EXPIRING",
               "TOTAL_GUARANTEED", "AVG_ANNUAL_VALUE", "BPM"]
present = [c for c in salary_cols if c in lakers.columns]

lakers_contracts = (
    lakers[present]
    .sort_values("SALARY", ascending=False)
    .reset_index(drop=True)
)
display(lakers_contracts)

total_payroll = lakers["SALARY"].sum()
total_guaranteed = lakers["TOTAL_GUARANTEED"].sum() if "TOTAL_GUARANTEED" in lakers.columns else None
expiring_count = lakers["IS_EXPIRING"].sum() if "IS_EXPIRING" in lakers.columns else None

print(f"\nTotal 2025-26 payroll:  ${total_payroll:,.0f}")
if total_guaranteed:
    print(f"Total guaranteed:       ${total_guaranteed:,.0f}")
if expiring_count is not None:
    print(f"Expiring contracts:     {int(expiring_count)}")

,PLAYER_NAME,SALARY,YEARS_REMAINING,IS_EXPIRING,TOTAL_GUARANTEED,AVG_ANNUAL_VALUE,BPM
0,LeBron James,52627153.00,1.00,True,52627153.00,52627153.00,3.50
1,Luka Dončić,45999660.00,4.00,False,149583660.00,51837915.00,9.30
2,Deandre Ayton,33654814.00,2.00,False,25550814.00,20879407.00,-0.90
3,Marcus Smart,19920855.00,2.00,False,14786855.00,12655777.50,-2.20
4,Rui Hachimura,18259259.00,1.00,True,18259259.00,18259259.00,-2.20
5,Austin Reaves,13937574.00,2.00,False,13937574.00,14418180.00,2.80
6,Jarred Vanderbilt,11571429.00,3.00,False,24000000.00,12428571.33,-1.50
7,Luke Kennard,11000000.00,1.00,True,11000000.00,11000000.00,0.10
8,Maxi Kleber,11000000.00,1.00,True,11000000.00,11000000.00,-3.90
9,Jake LaRavia,6000000.00,2.00,False,12000000.00,6000000.00,-1.20



Total 2025-26 payroll:  $244,804,828
Total guaranteed:       $358,954,819
Expiring contracts:     8


## 4. Top 20 by BPM and Surplus-Value Proxy

**Surplus proxy definition** (intentionally naive for now):
- Rank all players by BPM descending (`bpm_rank`: 1 = best BPM)
- Rank all players by SALARY descending (`salary_rank`: 1 = highest salary)
- `surplus_proxy = salary_rank - bpm_rank`
  - **Positive** = player produces above what their salary rank implies (undervalued)
  - **Negative** = player costs more than their production rank implies (overpaid)

This is a directional sanity check only. Phase 1 will replace this with a proper
surplus-value model using age-curve-adjusted projections.

In [6]:
# Filter to players with both BPM and SALARY populated
ranked = df.dropna(subset=["BPM", "SALARY"]).copy()
ranked = ranked[ranked["SALARY"] > 0].copy()

ranked["bpm_rank"] = ranked["BPM"].rank(ascending=False, method="min").astype(int)
ranked["salary_rank"] = ranked["SALARY"].rank(ascending=False, method="min").astype(int)
ranked["surplus_proxy"] = ranked["salary_rank"] - ranked["bpm_rank"]

display_cols = ["PLAYER_NAME", "TEAM_ABBREVIATION", "BPM", "SALARY", "bpm_rank", "salary_rank", "surplus_proxy"]
present = [c for c in display_cols if c in ranked.columns]

print("=== Top 20 by BPM ===")
display(ranked.nsmallest(20, "bpm_rank")[present].reset_index(drop=True))

print("\n=== Top 20 by Surplus Proxy (undervalued) ===")
display(ranked.nlargest(20, "surplus_proxy")[present].reset_index(drop=True))

=== Top 20 by BPM ===


,PLAYER_NAME,TEAM_ABBREVIATION,BPM,SALARY,bpm_rank,salary_rank,surplus_proxy
0,Nikola Jokić,DEN,14.20,55224526.00,1,2,1
1,Shai Gilgeous-Alexander,OKC,11.70,38333050.00,2,33,31
2,Victor Wembanyama,SAS,10.70,13376880.00,3,132,129
3,Giannis Antetokounmpo,MIL,9.50,54126450.00,4,5,1
4,Luka Dončić,LAL,9.30,45999660.00,5,22,17
5,Kawhi Leonard,LAC,8.00,50000000.00,6,14,8
6,Ty Jerome,MEM,7.70,8781000.00,7,181,174
7,Cade Cunningham,DET,6.30,46394100.00,8,16,8
8,Jimmy Butler III,GSW,5.50,54126450.00,9,5,-4
9,Stephen Curry,GSW,5.40,59606817.00,10,1,-9



=== Top 20 by Surplus Proxy (undervalued) ===


,PLAYER_NAME,TEAM_ABBREVIATION,BPM,SALARY,bpm_rank,salary_rank,surplus_proxy
0,Kadary Richmond,WAS,3.60,73153.00,31,463,432
1,Grant Nelson,BKN,2.20,73153.00,72,463,391
2,Charles Bassey,GSW,1.90,263940.00,87,453,366
3,Alondes Williams,WAS,1.10,131970.00,123,459,336
4,Myron Gardner,MIA,0.90,395029.00,129,451,322
5,Craig Porter Jr.,CLE,2.00,2221677.00,82,383,301
6,Skal Labissiere,WAS,0.30,131970.00,162,459,297
7,Neemias Queta,BOS,2.80,2349578.00,47,337,290
8,Collin Gillespie,PHX,2.50,2296274.00,60,345,285
9,Micah Potter,IND,0.70,1527805.00,138,423,285


## 5. Luka Doncic — Full Merged Row

Confirm all sources joined correctly for the player who anchors the entire Lakers analysis.

In [8]:
luka = df[df["PLAYER_NAME"].str.contains("Dončić", case=False, na=False)]
if luka.empty:
    print("Luka not found -- check name normalization or 2025-26 roster status")
else:
    display(luka.T.rename(columns={luka.index[0]: "Luka Doncic"}))

,Luka Doncic
PLAYER_ID,1629029
PLAYER_NAME,Luka Dončić
NICKNAME,Luka
TEAM_ID,1610612747
TEAM_ABBREVIATION,LAL
...,...
2026-27,49800000.00
2027-28,53784000.00
2028-29,57768000.00
2029-30,NaN


## Phase 1 Foundation — What We're Building On

Phase 0 is now complete. The merged dataset provides:

| Layer | Source | Key columns |
|-------|--------|-------------|
| Base stats | nba_api | PTS, AST, REB, FG%, 3P%, FT%, GP, MIN |
| Advanced (nba_api) | stats.nba.com | NET_RATING, USG_PCT, PIE, AST_PCT |
| Shooting splits | nba_api | CS_CATCH_SHOOT_FG3_PCT, pull-up rates |
| Defense | nba_api | PCT_PLUSMINUS (close-defense FG% diff) |
| Advanced (BBRef) | Basketball Reference | BPM, VORP, WS, WS/48, PER |
| Salary | BBRef | SALARY, YEARS_REMAINING, IS_EXPIRING, TOTAL_GUARANTEED |

**Phase 1 will build on this to:**
1. **Composite on-court value score** — weighted blend of BPM, WS/48, and nba_api NET_RATING
   (weights fitted by cross-validation against team win rate, not hand-tuned)
2. **Surplus value** — on-court value score vs. salary as % of cap; the inefficiency signal
3. **Age-curve adjustment** — project surplus over contract duration using historical decline curves
   by position/archetype (e.g., wings age differently than bigs)
4. **Durability weight** — scale surplus by DURABILITY_SCORE to discount injury-prone players
   (module already in `src/features/durability.py`)
5. **Backtest anchor** — validate the model against the 2023-24 Mavericks deadline:
   PJ Washington and Daniel Gafford should rank as high-surplus acquisitions prospectively

Key limitation to document: all metrics measure *output in context*, not isolated quality.
A player on a good team in a favorable role will show better numbers than the same player
in a worse situation. We correct for this partially via per-possession and rate stats;
we cannot fully eliminate system effects.